# 🏏 IPL Match Insights & Player Performance Analytics
### Exploratory Data Analysis Notebook
**Author:** Your Name  
**Dataset:** IPL Complete Dataset 2008–2020 (Kaggle)  
**Tech Stack:** Python · Pandas · NumPy · Matplotlib · Seaborn

## 📦 Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('✅ Libraries imported successfully!')

## 📂 Step 2 — Load Datasets

In [ ]:
# Load the two main IPL datasets
matches    = pd.read_csv('datasets/matches.csv')
deliveries = pd.read_csv('datasets/deliveries.csv')

print(f'matches.csv    → {matches.shape[0]:,} rows, {matches.shape[1]} columns')
print(f'deliveries.csv → {deliveries.shape[0]:,} rows, {deliveries.shape[1]} columns')

In [ ]:
# Preview matches dataset
matches.head(3)

In [ ]:
# Preview deliveries dataset
deliveries.head(3)

## 🧹 Step 3 — Data Cleaning

In [ ]:
# Check missing values
print('=== Missing Values in matches.csv ===')
print(matches.isnull().sum()[matches.isnull().sum() > 0])
print()
print('=== Missing Values in deliveries.csv ===')
print(deliveries.isnull().sum()[deliveries.isnull().sum() > 0])

In [ ]:
# ── Clean matches dataset ──────────────────────────────────────────

# 1. Drop duplicate rows
matches.drop_duplicates(inplace=True)

# 2. Fill missing values
matches['player_of_match'] = matches['player_of_match'].fillna('Unknown')
matches['winner']          = matches['winner'].fillna('No Result')
matches['city']            = matches['city'].fillna('Unknown')

# 3. Standardise old/renamed team names
team_map = {
    'Delhi Daredevils':        'Delhi Capitals',
    'Deccan Chargers':         'Sunrisers Hyderabad',
    'Pune Warriors':           'Rising Pune Supergiant',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Kings XI Punjab':         'Punjab Kings',
}
for col in ['team1', 'team2', 'winner', 'toss_winner']:
    matches[col] = matches[col].replace(team_map)

print(f'✅ matches cleaned → {matches.shape[0]} rows remaining')

In [ ]:
# ── Clean deliveries dataset ──────────────────────────────────────

deliveries.drop_duplicates(inplace=True)

# Fill numeric columns
for col in ['batsman_runs', 'extra_runs', 'total_runs', 'is_wicket']:
    deliveries[col] = pd.to_numeric(deliveries[col], errors='coerce').fillna(0)

# Standardise team names
for col in ['batting_team', 'bowling_team']:
    deliveries[col] = deliveries[col].replace(team_map)

print(f'✅ deliveries cleaned → {deliveries.shape[0]:,} rows remaining')

## 📊 Step 4 — Exploratory Data Analysis

### 4.1 Total Matches Won by Each Team

In [ ]:
wins = matches[matches['winner'] != 'No Result']['winner'].value_counts()

fig, ax = plt.subplots(figsize=(12, 6))
wins.plot(kind='barh', color='steelblue', ax=ax)
ax.set_title('🏆 Total IPL Matches Won by Each Team', fontsize=14)
ax.set_xlabel('Matches Won')
plt.tight_layout()
plt.savefig('screenshots/01_matches_won.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.2 Toss Impact on Match Result

In [ ]:
matches['toss_won_match'] = matches['toss_winner'] == matches['winner']
toss_counts = matches['toss_won_match'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
ax.pie(toss_counts.values,
       labels=['Toss ≠ Match Winner', 'Toss = Match Winner'],
       autopct='%1.1f%%',
       colors=['#E74C3C', '#2ECC71'],
       startangle=140)
ax.set_title('🎯 Does Winning the Toss Mean Winning the Match?')
plt.tight_layout()
plt.savefig('screenshots/02_toss_impact.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.3 Top 10 Run Scorers

In [ ]:
top_runs = deliveries.groupby('batsman')['batsman_runs'].sum()\
                     .sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_runs.plot(kind='bar', color='darkorange', ax=ax)
ax.set_title('🏏 Top 10 IPL Run Scorers (All Time)', fontsize=14)
ax.set_xlabel('Player')
ax.set_ylabel('Total Runs')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('screenshots/03_top_run_scorers.png', dpi=150, bbox_inches='tight')
plt.show()

print(top_runs)

### 4.4 Top 10 Wicket Takers

In [ ]:
wickets = deliveries[
    (deliveries['is_wicket'] == 1) &
    (deliveries['dismissal_kind'] != 'run out')
]
top_wickets = wickets.groupby('bowler')['is_wicket'].count()\
                     .sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(12, 6))
top_wickets.plot(kind='bar', color='purple', ax=ax)
ax.set_title('🎳 Top 10 IPL Wicket Takers (All Time)', fontsize=14)
ax.set_xlabel('Bowler')
ax.set_ylabel('Wickets')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('screenshots/04_top_wicket_takers.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Season-wise Average Score Trend (Line Graph)

In [ ]:
runs_per_match = deliveries.groupby('match_id')['total_runs'].sum().reset_index()
merged = runs_per_match.merge(matches[['id', 'season']], left_on='match_id', right_on='id')
season_avg = merged.groupby('season')['total_runs'].mean().round(1)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(season_avg.index, season_avg.values, marker='o', color='#E8871E', linewidth=2.5)
ax.fill_between(season_avg.index, season_avg.values, alpha=0.15, color='#E8871E')
ax.set_title('📈 Season-wise Average Match Score Trend', fontsize=14)
ax.set_xlabel('Season')
ax.set_ylabel('Average Total Score')
plt.tight_layout()
plt.savefig('screenshots/05_season_trend.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.6 Dismissal Heatmap

In [ ]:
wkt_df = deliveries[deliveries['is_wicket'] == 1]
heat = wkt_df.pivot_table(
    index='bowling_team', columns='dismissal_kind',
    values='is_wicket', aggfunc='count', fill_value=0
)

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(heat, annot=True, fmt='d', cmap='YlOrRd', ax=ax)
ax.set_title('🔥 Dismissal Type Heatmap by Bowling Team', fontsize=14)
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('screenshots/06_dismissal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.7 Orange Cap Winners

In [ ]:
meta = matches[['id', 'season']].rename(columns={'id': 'match_id'})
merged2 = deliveries.merge(meta, on='match_id')
season_runs = merged2.groupby(['season', 'batsman'])['batsman_runs'].sum().reset_index()
oc = season_runs.loc[season_runs.groupby('season')['batsman_runs'].idxmax()]

print('🟠 Orange Cap Winners')
print(oc[['season', 'batsman', 'batsman_runs']].to_string(index=False))

### 4.8 Purple Cap Winners

In [ ]:
wkts2 = deliveries[
    (deliveries['is_wicket'] == 1) &
    (deliveries['dismissal_kind'] != 'run out')
].merge(meta, on='match_id')

season_wkts = wkts2.groupby(['season', 'bowler'])['is_wicket'].count().reset_index()
pc = season_wkts.loc[season_wkts.groupby('season')['is_wicket'].idxmax()]

print('🟣 Purple Cap Winners')
print(pc[['season', 'bowler', 'is_wicket']].rename(columns={'is_wicket': 'wickets'}).to_string(index=False))

## ✅ Step 5 — Key Takeaways

| Insight | Finding |
|---|---|
| Most successful team | Mumbai Indians |
| Toss advantage | ~51% — minimal impact |
| Highest run scorer | V. Kohli |
| Highest wicket taker | SL Malinga |
| Favourite toss decision | Field first |

> **Next Step:** Open the Streamlit dashboard with `streamlit run app.py`